In [1]:
import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
#with open("../../config.yaml", "r") as f:
#    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import writing_tools as wt
import utils

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False


In [2]:
df = dt.get_user_by_week_panel(overwrite=False)
weekly_prices = dt.get_price_weekly()

c:\Users\edwar\projects\sn-research\src\notebooks\../python\data_tools.py:91: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time


In [3]:
weekly_prices['wow_growth'] = np.log(weekly_prices['btc_price']) - np.log(weekly_prices['btc_price'].shift(1))
weekly_prices['mom_growth'] = np.log(weekly_prices['btc_price']) - np.log(weekly_prices['btc_price'].shift(4))
weekly_prices['mom2_growth'] = np.log(weekly_prices['btc_price']) - np.log(weekly_prices['btc_price'].shift(8))
weekly_prices['yoy_growth'] = np.log(weekly_prices['btc_price']) - np.log(weekly_prices['btc_price'].shift(52))

In [4]:
# merge on weekly prices
df = df.merge(weekly_prices, on='week', how='left')

# remove SN employees
df = df.loc[~df['userId'].isin(globals.sn_employee_ids)].reset_index(drop=True)

# remove anon
df = df.loc[df['userId'] != globals.anon_id].reset_index(drop=True)

# remove territory owners
tdf = dt.get_territory_transfers()
df = df.loc[~df['userId'].isin(tdf['userIdTo'].unique())].reset_index(drop=True)

# remove weeks in which user is not active 
df['inactive'] = (df['weeks_since_last_activity']>=1) & (df['length_of_inactivity']>=4)
df['became_inactive'] = (df['weeks_since_last_activity']==1) & (df['length_of_inactivity']>=4)
df = df.loc[(~df['inactive']) | (df['became_inactive'])].reset_index(drop=True)

# output
df.to_parquet(os.path.join(DATA_PATH, "profitability-analysis.parquet"), index=False)

In [5]:
RESULTS = {
    "InactiveRateBaseline": f"{df['became_inactive'].mean()*100:.1f}"
}
_ = wt.update_results(RESULTS)
RESULTS

{'InactiveRateBaseline': '14.1'}

In [6]:
res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/05_profitability_analysis.R"], check=True, capture_output=True, text=True)
print(res.stdout)

                                           r1                  r2                  r3                  r4
Dependent Var.:               became_inactive     became_inactive     became_inactive     became_inactive
                                                                                                         
Constant                   0.0793*** (0.0019)  0.1565*** (0.0031)                                        
unprofitableTRUE           0.1053*** (0.0051)  0.0623*** (0.0047)  0.0601*** (0.0047)  0.0550*** (0.0049)
log_items                                     -0.0288*** (0.0008) -0.0291*** (0.0008) -0.0290*** (0.0008)
unprofitable_X_mom2_growth                                                              0.0561** (0.0184)
Fixed-Effects:             ------------------ ------------------- ------------------- -------------------
weekId                                     No                  No                 Yes                 Yes
__________________________ __________________ 

In [7]:
coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "profitability_analysis_coefs.parquet"))

idx = (coefs_df['regression_name']=='r4') & (coefs_df['coef_name']=='unprofitableTRUE')
coef1 = coefs_df.loc[idx, 'estimate'].values[0]

idx = (coefs_df['regression_name']=='r4') & (coefs_df['coef_name']=='unprofitable_X_mom2_growth')
coef2 = coefs_df.loc[idx, 'estimate'].values[0]

RESULTS = {
    'UnprofitableExitEffect': f"{coef1*100:.1f}", 
    'UnprofitableExitBTCGrowthEffect': f"{(coef1 + coef2*0.1)*100:.1f}"
}
_ = wt.update_results(RESULTS)
RESULTS

{'UnprofitableExitEffect': '5.5', 'UnprofitableExitBTCGrowthEffect': '6.1'}

In [8]:

header = r"""\begin{table}[H]
\centering
\caption{Effect of Profitability on User Exit} 
\vspace{0.2cm}
\label{tbl_profitability_analysis}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lcccc} 
\toprule
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
 & \multicolumn{4}{c}{Whether user becomes inactive} \\ [1ex]
 & (1) & (2) & (3) & (4) \\ 
\midrule
 &  &  &  &  \\
"""
footer = r"""\bottomrule
\end{tabular} 
\begin{tablenotes}[flushleft]\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:} Linear probability model of whether a user becomes inactive in a given week. A user becomes inactive if they start a four or more week consecutive period of no activity. The unit of observation is a user-week. Standard errors are clustered by user.
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r1", "r2", "r3", "r4"]

vars = [
    ("unprofitableTRUE", "Unprofitable in last 8 weeks"),
    ("unprofitable_X_mom2_growth", r"$\ldots \times$ BTC price appreciation in last 8 weeks"),
    ("log_items", "log(Items posted in last 8 weeks)"),
    ("(Intercept)", "Constant")
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1ex] " + "\n"

tbl += r"""& & & & \\
Week FE & N & N & Y & Y \\
 & & & & \\
"""

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ " + "\n"

tbl += r"R$^2$ "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="R2")
    r2 = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {r2:,.3f}"
tbl += r" \\ [1.8ex] " + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "results", "tbl_profitability_analysis.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)

\begin{table}[H]
\centering
\caption{Effect of Profitability on User Exit} 
\vspace{0.2cm}
\label{tbl_profitability_analysis}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lcccc} 
\toprule
 & \multicolumn{4}{c}{\textit{Dependent variable:}} \\ 
 & \multicolumn{4}{c}{Whether user becomes inactive} \\ [1ex]
 & (1) & (2) & (3) & (4) \\ 
\midrule
 &  &  &  &  \\
Unprofitable in last 8 weeks  & 0.105$^{***}$ & 0.062$^{***}$ & 0.060$^{***}$ & 0.055$^{***}$ \\
 & (0.005) & (0.005) & (0.005) & (0.005) \\ [1ex] 
$\ldots \times$ BTC price appreciation in last 8 weeks  &  &  &  & 0.056$^{***}$ \\
 &  &  &  & (0.018) \\ [1ex] 
log(Items posted in last 8 weeks)  &  & -0.029$^{***}$ & -0.029$^{***}$ & -0.029$^{***}$ \\
 &  & (0.001) & (0.001) & (0.001) \\ [1ex] 
Constant  & 0.079$^{***}$ & 0.157$^{***}$ &  &  \\
 & (0.002) & (0.003) &  &  \\ [1ex] 
& & & & \\
Week FE & N & N & Y & Y \\
 & & & & \\
Observations  & 92,956 & 92,956 & 92,956 & 92,956 \\ 
R$^2$  & 0.018 &